In [ ]:
!pip install -q transformers accelerate bitsandbytes
!pip install -q sentence-transformers language-tool-python
!pip install -q rouge-score sacrebleu nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.1 MB/s eta 0:00:00


In [ ]:
import torch
import nltk
nltk.download("punkt")

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer, util
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import sacrebleu
import language_tool_python

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [ ]:
from huggingface_hub import login
login()

In [ ]:
print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [ ]:
# Load LLM

model_name = "meta-llama/Llama-2-7b-chat-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading LLaMA 2...")

llm_tokenizer = AutoTokenizer.from_pretrained(model_name)

llm_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Model Loaded Successfully!")

Loading LLaMA 2...


config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

Model Loaded Successfully!


In [ ]:
# Load Embedding Model (Semantic Similarity)

semantic_model = SentenceTransformer("all-MiniLM-L6-v2")
grammar_tool = language_tool_python.LanguageTool('en-US')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def paraphrase_text(text):

    prompt = f"""[INST]
Paraphrase the sentence below.

Rules:
- Preserve all important information.
- Output only ONE rewritten sentence.
- Do NOT add explanations.
- Do NOT add commentary.
- Do NOT begin with words like "Sure" or "Here".

Sentence:
{text}
[/INST]"""

    inputs = llm_tokenizer(prompt, return_tensors="pt").to(llm_model.device)

    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=70,
            temperature=0.5,
            top_p=0.9,
            repetition_penalty=1.1,
            do_sample=True,
            eos_token_id=llm_tokenizer.eos_token_id
        )

    decoded = llm_tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract content after [/INST]
    result = decoded.split("[/INST]")[-1].strip()

    # Remove common assistant prefixes if present
    unwanted_prefixes = ["Sure", "Here", "Certainly", "Of course"]
    for prefix in unwanted_prefixes:
        if result.startswith(prefix):
            result = result.split("\n")[-1].strip()

    # Keep only first line
    result = result.split("\n")[0].strip().strip('"')

    return result

In [ ]:
# Grammar Correction
def grammar_check(text):
    matches = grammar_tool.check(text)
    corrected = language_tool_python.utils.correct(text, matches)
    return corrected

In [ ]:
# Evaluation Function

def evaluate(original, paraphrased):

    print("\n--- Evaluation ---")

    # BLEU (with smoothing)
    smooth = SmoothingFunction().method1
    bleu = sentence_bleu(
        [original.split()],
        paraphrased.split(),
        smoothing_function=smooth
    )
    print("BLEU:", round(bleu, 3))

    # ROUGE
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
    rouge_scores = scorer.score(original, paraphrased)
    print("ROUGE-1:", round(rouge_scores['rouge1'].fmeasure, 3))
    print("ROUGE-L:", round(rouge_scores['rougeL'].fmeasure, 3))

    # Semantic Similarity
    emb1 = semantic_model.encode(original, convert_to_tensor=True)
    emb2 = semantic_model.encode(paraphrased, convert_to_tensor=True)
    similarity = util.cos_sim(emb1, emb2)
    print("Semantic Similarity:", round(similarity.item(), 3))

    # SacreBLEU
    sacre = sacrebleu.corpus_bleu([paraphrased], [[original]])
    print("SacreBLEU:", round(sacre.score, 2))

In [ ]:
text = input("Enter text to paraphrase:\n")

raw_output = paraphrase_text(text)
final_output = grammar_check(raw_output)

print("\nOriginal:\n", text)
print("\nParaphrased:\n", final_output)

evaluate(text, final_output)

Enter text to paraphrase:
Artificial intelligence is transforming industries.

Original:
 Artificial intelligence is transforming industries.

Paraphrased:
 Artificial intelligence is revolutionizing various sectors.

--- Evaluation ---
BLEU: 0.202
ROUGE-1: 0.545
ROUGE-L: 0.545
Semantic Similarity: 0.822
SacreBLEU: 26.27
